IMPORTING MODULES

In [ ]:
import pandas as pd
import math

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving spam.csv to spam.csv


LOADING CSV FILE

In [ ]:
df = pd.read_csv("spam.csv", encoding="latin-1")

CHECKING FOR THE DATASET

In [ ]:
df.head()

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


In [ ]:
df.shape

(5572, 5)

In [ ]:
df.columns

Index(['v1', 'v2', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4'], dtype='object')

CHECKING NUMBER OF ZEROS

In [ ]:
df.isnull().sum()

,0
v1,0
v2,0
Unnamed: 2,5522
Unnamed: 3,5560
Unnamed: 4,5566


REMOVING UNNECESSARY COLUMNS

In [ ]:
df = df[['v1', 'v2']]

df.columns = ['label', 'message']

print(df.head())
print(df.shape)

  label                                            message
0   ham  Go until jurong point, crazy.. Available only ...
1   ham                      Ok lar... Joking wif u oni...
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...
3   ham  U dun say so early hor... U c already then say...
4   ham  Nah I don't think he goes to usf, he lives aro...
(5572, 2)


CHECKING THE LABELS

In [ ]:
print(df['label'].value_counts())

label
ham     4825
spam     747
Name: count, dtype: int64


CONVERTING LABLES INTO NUMBERS

In [ ]:
df['label'] = df['label'].map({
    'ham': 0,
    'spam': 1
})

print(df.head())
print(df['label'].value_counts())

   label                                            message
0      0  Go until jurong point, crazy.. Available only ...
1      0                      Ok lar... Joking wif u oni...
2      1  Free entry in 2 a wkly comp to win FA Cup fina...
3      0  U dun say so early hor... U c already then say...
4      0  Nah I don't think he goes to usf, he lives aro...
label
0    4825
1     747
Name: count, dtype: int64


SEPERATING X AND Y

In [ ]:
X = df['message']
y = df['label']

print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nFirst 5 messages:")
print(X.head())

print("\nFirst 5 labels:")
print(y.head())

X shape: (5572,)
y shape: (5572,)

First 5 messages:
0    Go until jurong point, crazy.. Available only ...
1                        Ok lar... Joking wif u oni...
2    Free entry in 2 a wkly comp to win FA Cup fina...
3    U dun say so early hor... U c already then say...
4    Nah I don't think he goes to usf, he lives aro...
Name: message, dtype: object

First 5 labels:
0    0
1    0
2    1
3    0
4    0
Name: label, dtype: int64


TRAIN-TEST-SPLIT

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training messages:", X_train.shape)
print("Testing messages:", X_test.shape)
print("Training labels:", y_train.shape)
print("Testing labels:", y_test.shape)

Training messages: (4457,)
Testing messages: (1115,)
Training labels: (4457,)
Testing labels: (1115,)


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

TF-IDF

In [ ]:
tfidf = TfidfVectorizer()

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)
print("Training TF-IDF shape:", X_train_tfidf.shape)
print("Testing TF-IDF shape:", X_test_tfidf.shape)

Training TF-IDF shape: (4457, 7701)
Testing TF-IDF shape: (1115, 7701)


TRAINING 3 MODELS

In [ ]:
nb_model = MultinomialNB()

nb_model.fit(X_train_tfidf, y_train)

nb_pred = nb_model.predict(X_test_tfidf)

In [ ]:
lr_model = LogisticRegression(max_iter=1000)

lr_model.fit(X_train_tfidf, y_train)

lr_pred = lr_model.predict(X_test_tfidf)

In [ ]:
svm_model = LinearSVC()

svm_model.fit(X_train_tfidf, y_train)

svm_pred = svm_model.predict(X_test_tfidf)

COMPARE MODELS

In [ ]:
nb_accuracy = accuracy_score(y_test, nb_pred)
lr_accuracy = accuracy_score(y_test, lr_pred)
svm_accuracy = accuracy_score(y_test, svm_pred)

print("Naive Bayes Accuracy:", nb_accuracy * 100, "%")
print("Logistic Regression Accuracy:", lr_accuracy * 100, "%")
print("SVM Accuracy:", svm_accuracy * 100, "%")

Naive Bayes Accuracy: 96.05381165919282 %
Logistic Regression Accuracy: 97.21973094170404 %
SVM Accuracy: 98.47533632286995 %


In [ ]:
print("===== Naive Bayes =====")
print(classification_report(
    y_test, nb_pred,
    target_names=['Not Spam', 'Spam']
))

print("===== Logistic Regression =====")
print(classification_report(
    y_test, lr_pred,
    target_names=['Not Spam', 'Spam']
))

print("===== SVM =====")
print(classification_report(
    y_test, svm_pred,
    target_names=['Not Spam', 'Spam']
))

===== Naive Bayes =====
              precision    recall  f1-score   support

    Not Spam       0.96      1.00      0.98       966
        Spam       1.00      0.70      0.83       149

    accuracy                           0.96      1115
   macro avg       0.98      0.85      0.90      1115
weighted avg       0.96      0.96      0.96      1115

===== Logistic Regression =====
              precision    recall  f1-score   support

    Not Spam       0.97      1.00      0.98       966
        Spam       0.99      0.80      0.88       149

    accuracy                           0.97      1115
   macro avg       0.98      0.90      0.93      1115
weighted avg       0.97      0.97      0.97      1115

===== SVM =====
              precision    recall  f1-score   support

    Not Spam       0.98      1.00      0.99       966
        Spam       0.99      0.90      0.94       149

    accuracy                           0.98      1115
   macro avg       0.98      0.95      0.97      1115
we

MAIN MODEL

In [ ]:
model = svm_model

SUSPICIOUS WORDS

In [ ]:
suspicious_words = {
    "urgent": [
        "urgent", "immediately", "act now", "limited time"
    ],

    "prize": [
        "winner", "won", "prize", "reward",
        "free", "congratulations"
    ],

    "money": [
        "cash", "money", "loan", "profit",
        "₹", "rs", "rupees"
    ],

    "personal_information": [
        "password", "otp", "bank details",
        "account number", "pin"
    ],

    "suspicious_link": [
        "click", "http", "https", "www", "link"
    ]
}

In [ ]:
def find_reasons(message):

    message_lower = message.lower()
    reasons = []

    for category, words in suspicious_words.items():

        found_words = []

        for word in words:
            if word in message_lower:
                found_words.append(word)

        if found_words:
            reasons.append((category, found_words))

    return reasons

FINAL SVM CHECKER

In [ ]:
def check_message_svm(message):

    # Convert message to TF-IDF
    message_tfidf = tfidf.transform([message])

    # SVM prediction
    prediction = svm_model.predict(message_tfidf)[0]

    # SVM decision score
    decision_score = svm_model.decision_function(message_tfidf)[0]

    # Convert score to model-based risk score
    spam_score = 1 / (1 + math.exp(-decision_score))

    risk_score = round(spam_score * 100)

    # Risk level
    if risk_score <= 30:
        risk_level = "Low"
    elif risk_score <= 70:
        risk_level = "Medium"
    else:
        risk_level = "High"

    # Prediction name
    if prediction == 1:
        result = "Spam"
    else:
        result = "Not Spam"

    # Find reasons
    reasons = find_reasons(message)

    # Display result
    print("Message:", message)
    print("Prediction:", result)
    print("SVM Decision Score:", round(decision_score, 2))
    print("Risk Score:", risk_score, "/100")
    print("Risk Level:", risk_level)

    print("\nWhy is this message risky?")

    if reasons:
        for category, words in reasons:
            print("-", category, ":", ", ".join(words))
    else:
        print("- No suspicious keywords detected")

TEST

In [ ]:
check_message_svm(
    "Congratulations! You won a free iPhone. Click now to claim your prize!"
)

Message: Congratulations! You won a free iPhone. Click now to claim your prize!
Prediction: Spam
SVM Decision Score: 1.02
Risk Score: 74 /100
Risk Level: High

Why is this message risky?
- prize : won, prize, free, congratulations
- suspicious_link : click
